<a href="https://colab.research.google.com/github/drfperez/utilities/blob/main/Corrector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import cv2
import numpy as np
import zipfile
import os
import csv
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

# ==========================================
# 1. GENERADOR D'IMATGES (Amb ID Alumne)
# ==========================================
def generar_plantilla(tipus_plantilla, nom_arxiu, pauta_correcta=None, id_alumne=27):
    scale = 5
    width, height = 1050, 1485
    img = Image.new('RGB', (width, height), color='white')
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype("LiberationSans-Bold.ttf", 30)
        font_text = ImageFont.truetype("LiberationSans-Regular.ttf", 18)
        font_small = ImageFont.truetype("LiberationSans-Regular.ttf", 14)
    except:
        font_title = ImageFont.load_default()
        font_text = ImageFont.load_default()
        font_small = ImageFont.load_default()

    # 1. CAPÇALERA
    linia_dades = "Nom: __________________________________   Assignatura: ______________________   Grup: _____   Data: _________"
    draw.text((75, 50), linia_dades, fill='black', font=font_text)

    titols = {
        'buida': "FULL DE RESPOSTES",
        'profe': "PAUTA DE CORRECCIÓ (PROFESSOR)",
        'alumne': "EXAMEN D'ALUMNE (EXEMPLE AMB ERRORS I ID)"
    }
    draw.text((75, 100), titols[tipus_plantilla], fill='black', font=font_title)
    draw.line([(75, 140), (975, 140)], fill='black', width=2)

    # 2. MARQUES DE POSICIONAMENT (Escàner)
    m_size = 6 * scale
    marques = [(15*scale, 40*scale), (189*scale, 40*scale), (15*scale, 275*scale), (189*scale, 275*scale)]
    for x, y in marques:
        draw.rectangle([x, y, x + m_size, y + m_size], fill='black')

    box_w, box_h = 6 * scale, 3 * scale

    # 3. BLOC D'IDENTIFICACIÓ DE L'ALUMNE (00 al 39)
    draw.text((130, 175), "ID Alumne - Desenes:", fill='black', font=font_small)
    x_start_id = 320
    for i in range(4): # Del 0 al 3
        x, y = x_start_id + i * 40, 175
        draw.text((x + 10, y - 18), str(i), fill='black', font=font_small)
        fill_c = '#334155' if tipus_plantilla == 'alumne' and id_alumne is not None and (id_alumne // 10) == i else None
        draw.rectangle([x, y, x + box_w, y + box_h], outline='black', fill=fill_c, width=1)

    draw.text((130, 220), "ID Alumne - Unitats:", fill='black', font=font_small)
    for i in range(10): # Del 0 al 9
        x, y = x_start_id + i * 40, 220
        draw.text((x + 10, y - 18), str(i), fill='black', font=font_small)
        fill_c = '#334155' if tipus_plantilla == 'alumne' and id_alumne is not None and (id_alumne % 10) == i else None
        draw.rectangle([x, y, x + box_w, y + box_h], outline='black', fill=fill_c, width=1)


    # 4. CONFIGURAR LES RESPOSTES
    lletres = ['A', 'B', 'C', 'D']
    respostes = [''] * 100
    if tipus_plantilla == 'profe' or tipus_plantilla == 'alumne':
        respostes = list(pauta_correcta)

    if tipus_plantilla == 'alumne':
        indexs_errors = [2, 18, 35, 55, 72, 88]
        indexs_blancs = [10, 45, 95]
        for i in indexs_errors:
            respostes[i] = 'B' if respostes[i] == 'A' else 'A'
        for i in indexs_blancs:
            respostes[i] = ''

    # 5. DIBUIXAR PREGUNTES (1 a 100)
    x_cols = [[35, 45, 55, 65], [115, 125, 135, 145]]

    y_lletres = int(49 * scale)
    for col_idx in range(2):
        for c in range(4):
            x_lletra = int((x_cols[col_idx][c] + 1.5) * scale)
            draw.text((x_lletra, y_lletres), lletres[c], fill='black', font=font_small)

    for i in range(50):
        y = int((53 + i * (215 / 49)) * scale)
        for col_idx, num_pregunta in enumerate([i, i + 50]):
            x_num = int((x_cols[col_idx][0] - 6) * scale)
            draw.text((x_num, y), f"{num_pregunta + 1}", fill='black', font=font_small)
            for c in range(4):
                x = x_cols[col_idx][c] * scale
                fill_color = '#334155' if respostes[num_pregunta] == lletres[c] else None
                draw.rectangle([x, y, x + box_w, y + box_h], outline='black', fill=fill_color, width=1)

    img.save(nom_arxiu)
    return nom_arxiu

# ==========================================
# 2. MOTOR DEL CORRECTOR (Lector OMR)
# ==========================================
def extreure_respostes_imatge(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, img_thresh = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY_INV)

    scale = 5
    box_w, box_h = 30, 15

    # 1. LLEGIR ID DE L'ALUMNE
    desenes, unitats = -1, -1
    x_start_id = 320

    for i in range(4): # Llegir Desenes
        x, y = x_start_id + i * 40, 175
        if cv2.countNonZero(img_thresh[y:y+box_h, x:x+box_w]) > 100:
            desenes = i

    for i in range(10): # Llegir Unitats
        x, y = x_start_id + i * 40, 220
        if cv2.countNonZero(img_thresh[y:y+box_h, x:x+box_w]) > 100:
            unitats = i

    id_alumne = (desenes * 10 + unitats) if (desenes != -1 and unitats != -1) else -1

    # 2. LLEGIR RESPOSTES (1-100)
    x_cols = [[35, 45, 55, 65], [115, 125, 135, 145]]
    lletres = ['A', 'B', 'C', 'D']
    respostes = []

    for col_offset in [0, 50]:
        col_idx = 0 if col_offset == 0 else 1
        for i in range(50):
            y = int((53 + i * (215 / 49)) * scale)
            opcions = []
            for c in range(4):
                x = int(x_cols[col_idx][c] * scale)
                roi = img_thresh[y:y+box_h, x:x+box_w]
                if cv2.countNonZero(roi) > 100:
                    opcions.append(lletres[c])
            respostes.append(opcions[0] if len(opcions) == 1 else ('MULTIPLE' if len(opcions) > 1 else ''))

    return respostes, id_alumne

def corregir_examen(pauta_path, alumne_path, alumne_nom, penalitzacio=0.25):
    resp_profe, _ = extreure_respostes_imatge(pauta_path)
    resp_alumne, id_alumne = extreure_respostes_imatge(alumne_path)

    encerts, errors, blancs = 0, 0, 0
    for i in range(100):
        if resp_alumne[i] == '':
            blancs += 1
        elif resp_alumne[i] == resp_profe[i]:
            encerts += 1
        else:
            errors += 1

    puntuacio_neta = encerts - (errors * penalitzacio)
    nota_sobre_10 = max(0, (puntuacio_neta / 100) * 10)

    id_text = f"{id_alumne:02d}" if id_alumne != -1 else "NO MARCAT"

    print(f"\n📄 Correcció de l'arxiu: {alumne_nom}")
    print(f"👤 NÚMERO D'ALUMNE (ID): {id_text}")
    print("-" * 45)
    print(f"✅ Encerts: {encerts} | ❌ Errors: {errors} | ⚪ Blancs: {blancs}")
    print(f"🎯 NOTA FINAL: {nota_sobre_10:.2f} / 10")
    print("=" * 45)

    # Retornem les dades per poder generar el CSV
    return {
        'id_alumne': id_text,
        'nota': nota_sobre_10,
        'encerts': encerts,
        'errors': errors,
        'blancs': blancs,
        'arxiu': alumne_nom
    }

# ==========================================
# 3. INTERFÍCIE PER A L'USUARI
# ==========================================
print("=== SISTEMA OMR: CORRECCIÓ + IDENTIFICADOR ALUMNE ===")
print("1. Generar exàmens de prova (Genera l'alumne nº 27 com a exemple)")
print("2. Pujar Pauta i Corregir diversos alumnes (Fitxer a fitxer)")
print("3. NOU: Pujar ZIP amb exàmens i generar CSV amb les notes")
opcio = input("Tria una opció (1, 2 o 3): ")

if opcio == '1':
    input_pauta = input("Pauta (ex: ABCD...). Deixa en blanc per defecte: ").strip().upper()
    lletres_valides = ['A', 'B', 'C', 'D']
    pauta = [c for c in input_pauta if c in lletres_valides]
    while len(pauta) < 100:
        pauta.append(lletres_valides[len(pauta) % 4])
    pauta = pauta[:100]

    f1 = generar_plantilla('buida', '1_Plantilla_Buida.png')
    f2 = generar_plantilla('profe', '2_Pauta_Professor.png', pauta)
    f3 = generar_plantilla('alumne', '3_Exemple_Alumne_ID27.png', pauta, id_alumne=27)

    files.download(f1)
    files.download(f2)
    files.download(f3)
    print("\n✅ Plantilles generades i descarregades.")

    print("\n🔍 Simulació de correcció de l'exemple generat (Penalització -0.25):")
    corregir_examen(f2, f3, "3_Exemple_Alumne_ID27.png", penalitzacio=0.25)

elif opcio == '2':
    penalitzacio = float(input("Introdueix la penalització per error (ex: 0.25, 0.33, 0): ") or "0.25")

    print("\n[1/2] 📸 Puja la imatge de la PAUTA DEL PROFESSOR (1 arxiu):")
    pujada_pauta = files.upload()

    if pujada_pauta:
        nom_pauta = list(pujada_pauta.keys())[0]
        print("\n" + "="*45)
        print(" 📊 INICIANT CORRECCIONS")
        print("="*45)

        continuar = True
        while continuar:
            print("\n[2/2] 📸 Puja EXÀMENS D'ALUMNES.")
            pujada_alumnes = files.upload()

            if pujada_alumnes:
                for arxiu_alumne in pujada_alumnes.keys():
                    corregir_examen(nom_pauta, arxiu_alumne, arxiu_alumne, penalitzacio)
            else:
                print("⚠️ No s'ha pujat cap arxiu en aquest torn.")

            resposta = input("\n➡️ Vols pujar MÉS exàmens per corregir? (s/n): ").strip().lower()
            if resposta != 's':
                continuar = False
                print("\n✅ Procés de correcció finalitzat.")

    else:
        print("❌ No s'ha pujat la pauta. Torna a executar la cel·la.")

elif opcio == '3':
    penalitzacio = float(input("Introdueix la penalització per error (ex: 0.25, 0.33, 0): ") or "0.25")

    print("\n[1/2] 📸 Puja la imatge de la PAUTA DEL PROFESSOR (1 arxiu):")
    pujada_pauta = files.upload()

    if pujada_pauta:
        nom_pauta = list(pujada_pauta.keys())[0]

        print("\n[2/2] 📦 Puja l'arxiu ZIP que conté tots els EXÀMENS D'ALUMNES (imatges):")
        pujada_zip = files.upload()

        if pujada_zip:
            nom_zip = list(pujada_zip.keys())[0]
            dir_extracci = "examens_alumnes_zip"

            # Descomprimir l'arxiu
            os.makedirs(dir_extracci, exist_ok=True)
            with zipfile.ZipFile(nom_zip, 'r') as zip_ref:
                zip_ref.extractall(dir_extracci)

            print("\n" + "="*45)
            print(" 📊 CORREGINT EXÀMENS DEL ZIP...")
            print("="*45)

            resultats = []
            formats_valids = ('.png', '.jpg', '.jpeg', '.bmp')

            # Recórrer les imatges descomprimides i corregir-les
            for root, dirs, arxius in os.walk(dir_extracci):
                for arxiu in arxius:
                    if arxiu.lower().endswith(formats_valids):
                        ruta_completa = os.path.join(root, arxiu)
                        try:
                            dades_alumne = corregir_examen(nom_pauta, ruta_completa, arxiu, penalitzacio)
                            resultats.append(dades_alumne)
                        except Exception as e:
                            print(f"⚠️ No s'ha pogut processar {arxiu}: {e}")

            # Crear l'arxiu CSV
            nom_csv = "notes_alumnes.csv"
            with open(nom_csv, mode='w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f, delimiter=';') # Utilitzem ; per excel europeu
                writer.writerow(['ID_Alumne', 'Nota_Sobre_10', 'Encerts', 'Errors', 'Blancs', 'Nom_Arxiu'])

                # Ordenar per ID alumne per fer-ho més bonic al CSV
                resultats.sort(key=lambda x: str(x['id_alumne']))

                for r in resultats:
                    # Formategem la nota canviant el punt per la coma per llegir a l'Excel
                    nota_format = f"{r['nota']:.2f}".replace('.', ',')
                    writer.writerow([r['id_alumne'], nota_format, r['encerts'], r['errors'], r['blancs'], r['arxiu']])

            print(f"\n✅ S'han corregit {len(resultats)} exàmens correctament.")
            print(f"📁 Generant arxiu amb les notes: {nom_csv} ...")

            # Descarregar automàticament el CSV
            files.download(nom_csv)
        else:
            print("❌ No s'ha pujat cap arxiu ZIP. Torna a executar la cel·la.")
    else:
        print("❌ No s'ha pujat la pauta. Torna a executar la cel·la.")

else:
    print("❌ Opció no vàlida.")

In [10]:

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

# ==========================================
# 1. GENERADOR D'IMATGES (Amb ID Alumne)
# ==========================================
def generar_plantilla(tipus_plantilla, nom_arxiu, pauta_correcta=None, id_alumne=27):
    scale = 5
    width, height = 1050, 1485
    img = Image.new('RGB', (width, height), color='white')
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype("LiberationSans-Bold.ttf", 30)
        font_text = ImageFont.truetype("LiberationSans-Regular.ttf", 18)
        font_small = ImageFont.truetype("LiberationSans-Regular.ttf", 14)
    except:
        font_title = ImageFont.load_default()
        font_text = ImageFont.load_default()
        font_small = ImageFont.load_default()

    # 1. CAPÇALERA
    linia_dades = "Nom: __________________________________   Assignatura: ______________________   Grup: _____   Data: _________"
    draw.text((75, 50), linia_dades, fill='black', font=font_text)

    titols = {
        'buida': "FULL DE RESPOSTES",
        'profe': "PAUTA DE CORRECCIÓ (PROFESSOR)",
        'alumne': "EXAMEN D'ALUMNE (EXEMPLE AMB ERRORS I ID)"
    }
    draw.text((75, 100), titols[tipus_plantilla], fill='black', font=font_title)
    draw.line([(75, 140), (975, 140)], fill='black', width=2)

    # 2. MARQUES DE POSICIONAMENT (Escàner)
    m_size = 6 * scale
    marques = [(15*scale, 40*scale), (189*scale, 40*scale), (15*scale, 275*scale), (189*scale, 275*scale)]
    for x, y in marques:
        draw.rectangle([x, y, x + m_size, y + m_size], fill='black')

    box_w, box_h = 6 * scale, 3 * scale

    # 3. BLOC D'IDENTIFICACIÓ DE L'ALUMNE (00 al 39)
    # Desplaçat a x=130 per no xocar amb la marca negra de l'escàner de l'esquerra
    draw.text((130, 175), "ID Alumne - Desenes:", fill='black', font=font_small)
    x_start_id = 320
    for i in range(4): # Del 0 al 3
        x, y = x_start_id + i * 40, 175
        draw.text((x + 10, y - 18), str(i), fill='black', font=font_small)
        # Pintar només si és un examen d'alumne i coincideix
        fill_c = '#334155' if tipus_plantilla == 'alumne' and id_alumne is not None and (id_alumne // 10) == i else None
        draw.rectangle([x, y, x + box_w, y + box_h], outline='black', fill=fill_c, width=1)

    draw.text((130, 220), "ID Alumne - Unitats:", fill='black', font=font_small)
    for i in range(10): # Del 0 al 9
        x, y = x_start_id + i * 40, 220
        draw.text((x + 10, y - 18), str(i), fill='black', font=font_small)
        fill_c = '#334155' if tipus_plantilla == 'alumne' and id_alumne is not None and (id_alumne % 10) == i else None
        draw.rectangle([x, y, x + box_w, y + box_h], outline='black', fill=fill_c, width=1)


    # 4. CONFIGURAR LES RESPOSTES
    lletres = ['A', 'B', 'C', 'D']
    respostes = [''] * 100
    if tipus_plantilla == 'profe' or tipus_plantilla == 'alumne':
        respostes = list(pauta_correcta)

    if tipus_plantilla == 'alumne':
        # Introduir exactament 6 errors i 3 blancs
        indexs_errors = [2, 18, 35, 55, 72, 88]
        indexs_blancs = [10, 45, 95]
        for i in indexs_errors:
            respostes[i] = 'B' if respostes[i] == 'A' else 'A'
        for i in indexs_blancs:
            respostes[i] = ''

    # 5. DIBUIXAR PREGUNTES (1 a 100)
    x_cols = [[35, 45, 55, 65], [115, 125, 135, 145]]

    y_lletres = int(49 * scale)
    for col_idx in range(2):
        for c in range(4):
            x_lletra = int((x_cols[col_idx][c] + 1.5) * scale)
            draw.text((x_lletra, y_lletres), lletres[c], fill='black', font=font_small)

    for i in range(50):
        y = int((53 + i * (215 / 49)) * scale)
        for col_idx, num_pregunta in enumerate([i, i + 50]):
            x_num = int((x_cols[col_idx][0] - 6) * scale)
            draw.text((x_num, y), f"{num_pregunta + 1}", fill='black', font=font_small)
            for c in range(4):
                x = x_cols[col_idx][c] * scale
                fill_color = '#334155' if respostes[num_pregunta] == lletres[c] else None
                draw.rectangle([x, y, x + box_w, y + box_h], outline='black', fill=fill_color, width=1)

    img.save(nom_arxiu)
    return nom_arxiu

# ==========================================
# 2. MOTOR DEL CORRECTOR (Lector OMR)
# ==========================================
def extreure_respostes_imatge(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    _, img_thresh = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY_INV)

    scale = 5
    box_w, box_h = 30, 15

    # 1. LLEGIR ID DE L'ALUMNE
    desenes, unitats = -1, -1
    x_start_id = 320

    for i in range(4): # Llegir Desenes
        x, y = x_start_id + i * 40, 175
        if cv2.countNonZero(img_thresh[y:y+box_h, x:x+box_w]) > 100:
            desenes = i

    for i in range(10): # Llegir Unitats
        x, y = x_start_id + i * 40, 220
        if cv2.countNonZero(img_thresh[y:y+box_h, x:x+box_w]) > 100:
            unitats = i

    id_alumne = (desenes * 10 + unitats) if (desenes != -1 and unitats != -1) else -1

    # 2. LLEGIR RESPOSTES (1-100)
    x_cols = [[35, 45, 55, 65], [115, 125, 135, 145]]
    lletres = ['A', 'B', 'C', 'D']
    respostes = []

    for col_offset in [0, 50]:
        col_idx = 0 if col_offset == 0 else 1
        for i in range(50):
            y = int((53 + i * (215 / 49)) * scale)
            opcions = []
            for c in range(4):
                x = int(x_cols[col_idx][c] * scale)
                roi = img_thresh[y:y+box_h, x:x+box_w]
                if cv2.countNonZero(roi) > 100:
                    opcions.append(lletres[c])
            respostes.append(opcions[0] if len(opcions) == 1 else ('MULTIPLE' if len(opcions) > 1 else ''))

    return respostes, id_alumne

def corregir_examen(pauta_path, alumne_path, alumne_nom, penalitzacio=0.25):
    resp_profe, _ = extreure_respostes_imatge(pauta_path)
    resp_alumne, id_alumne = extreure_respostes_imatge(alumne_path)

    encerts, errors, blancs = 0, 0, 0
    for i in range(100):
        if resp_alumne[i] == '':
            blancs += 1
        elif resp_alumne[i] == resp_profe[i]:
            encerts += 1
        else:
            errors += 1

    puntuacio_neta = encerts - (errors * penalitzacio)
    nota_sobre_10 = max(0, (puntuacio_neta / 100) * 10)

    # Formatejar el número de l'alumne (ex: 05, 27)
    id_text = f"{id_alumne:02d}" if id_alumne != -1 else "NO MARCAT O LLEGIT INCORRECTAMENT"

    print(f"\n📄 Correcció de l'arxiu: {alumne_nom}")
    print(f"👤 NÚMERO D'ALUMNE (ID): {id_text}")
    print("-" * 45)
    print(f"✅ Encerts: {encerts} | ❌ Errors: {errors} | ⚪ Blancs: {blancs}")
    print(f"🎯 NOTA FINAL: {nota_sobre_10:.2f} / 10")
    print("=" * 45)

# ==========================================
# 3. INTERFÍCIE PER A L'USUARI
# ==========================================
print("=== SISTEMA OMR: CORRECCIÓ + IDENTIFICADOR ALUMNE ===")
print("1. Generar exàmens de prova (Genera l'alumne nº 27 com a exemple)")
print("2. Pujar Pauta i Corregir diversos alumnes")
opcio = input("Tria una opció (1 o 2): ")

if opcio == '1':
    input_pauta = input("Pauta (ex: ABCD...). Deixa en blanc per defecte: ").strip().upper()
    lletres_valides = ['A', 'B', 'C', 'D']
    pauta = [c for c in input_pauta if c in lletres_valides]
    while len(pauta) < 100:
        pauta.append(lletres_valides[len(pauta) % 4])
    pauta = pauta[:100]

    f1 = generar_plantilla('buida', '1_Plantilla_Buida.png')
    f2 = generar_plantilla('profe', '2_Pauta_Professor.png', pauta)
    f3 = generar_plantilla('alumne', '3_Exemple_Alumne_ID27.png', pauta, id_alumne=27)

    files.download(f1)
    files.download(f2)
    files.download(f3)
    print("\n✅ Plantilles generades i descarregades.")

    print("\n🔍 Simulació de correcció de l'exemple generat (Penalització -0.25):")
    corregir_examen(f2, f3, "3_Exemple_Alumne_ID27.png", penalitzacio=0.25)

elif opcio == '2':
    penalitzacio = float(input("Introdueix la penalització per error (ex: 0.25, 0.33, 0): ") or "0.25")

    print("\n[1/2] 📸 Puja la imatge de la PAUTA DEL PROFESSOR (1 arxiu):")
    pujada_pauta = files.upload()

    if pujada_pauta:
        nom_pauta = list(pujada_pauta.keys())[0]

        print("\n" + "="*45)
        print(" 📊 INICIANT CORRECCIONS")
        print("="*45)

        continuar = True
        while continuar:
            print("\n[2/2] 📸 Puja EXÀMENS D'ALUMNES.")
            print("💡 Consell: Pots seleccionar-ne varis de cop mantenint premuda la tecla Ctrl o Cmd.")
            pujada_alumnes = files.upload()

            if pujada_alumnes:
                for arxiu_alumne in pujada_alumnes.keys():
                    corregir_examen(nom_pauta, arxiu_alumne, arxiu_alumne, penalitzacio)
            else:
                print("⚠️ No s'ha pujat cap arxiu en aquest torn.")

            resposta = input("\n➡️ Vols pujar MÉS exàmens per corregir? (s/n): ").strip().lower()
            if resposta != 's':
                continuar = False
                print("\n✅ Procés de correcció finalitzat.")

    else:
        print("❌ No s'ha pujat la pauta. Torna a executar la cel·la.")
else:
    print("Opció no vàlida.")

=== SISTEMA OMR: CORRECCIÓ + IDENTIFICADOR ALUMNE ===
1. Generar exàmens de prova (Genera l'alumne nº 27 com a exemple)
2. Pujar Pauta i Corregir diversos alumnes
Tria una opció (1 o 2): 2
Introdueix la penalització per error (ex: 0.25, 0.33, 0): 

[1/2] 📸 Puja la imatge de la PAUTA DEL PROFESSOR (1 arxiu):


Saving 2_Pauta_Professor.png to 2_Pauta_Professor (4).png

 📊 INICIANT CORRECCIONS

[2/2] 📸 Puja EXÀMENS D'ALUMNES.
💡 Consell: Pots seleccionar-ne varis de cop mantenint premuda la tecla Ctrl o Cmd.


Saving 3_Exemple_Alumne_ID27.png to 3_Exemple_Alumne_ID27 (4).png

📄 Correcció de l'arxiu: 3_Exemple_Alumne_ID27 (4).png
👤 NÚMERO D'ALUMNE (ID): 27
---------------------------------------------
✅ Encerts: 91 | ❌ Errors: 6 | ⚪ Blancs: 3
🎯 NOTA FINAL: 8.95 / 10

➡️ Vols pujar MÉS exàmens per corregir? (s/n): S

[2/2] 📸 Puja EXÀMENS D'ALUMNES.
💡 Consell: Pots seleccionar-ne varis de cop mantenint premuda la tecla Ctrl o Cmd.


Saving 3_Exemple_Alumne_ID27.png to 3_Exemple_Alumne_ID27 (5).png

📄 Correcció de l'arxiu: 3_Exemple_Alumne_ID27 (5).png
👤 NÚMERO D'ALUMNE (ID): 27
---------------------------------------------
✅ Encerts: 91 | ❌ Errors: 6 | ⚪ Blancs: 3
🎯 NOTA FINAL: 8.95 / 10

➡️ Vols pujar MÉS exàmens per corregir? (s/n): N

✅ Procés de correcció finalitzat.
